In [1]:
# WS H Setup — load data, re-derive lever targets, define helper functions
# (Recreates WS G state for the new notebook kernel.)

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# --- Load loan-level data (same as WS G Cell 1) ---
# Canonical flattened path (WS E nesting removed during reconciliation).
DATA_DIR = "../WS E - Customer Segmentation"
lf = pd.read_parquet(f"{DATA_DIR}/out/frames/loan_frame.parquet")
seg = pd.read_csv(f"{DATA_DIR}/step3_segment_assignments.csv")
lf = lf.merge(seg[["loan_id", "segment"]], on="loan_id")
print(f"Loaded {len(lf):,} loans with segments\n")

# --- Locked, data-generating assumptions (config.py, A-044) ---
# These are the TRUE LGDs used to generate value_proxy. Earlier drafts of this
# notebook used 0.60/0.60/0.50, which did not match the data; corrected below.
LGD_BNPL = 0.40       # BNPL (A-044)
LGD_PERSONAL = 0.65   # Personal — unsecured retail (A-044)
LGD_SME = 0.75        # SME (A-044)

# --- Re-derive lever target sets ---
bnpl = lf[lf["product_type"] == "BNPL"]
lever1_targets = bnpl[bnpl["acquisition_channel"].isin(["Digital ads", "DSA"])]

ppl_de = lf[(lf["product_type"] == "Personal") & 
            (lf["origination_risk_grade"].isin(["D", "E"]))]
lever2_targets = ppl_de[ppl_de["cashflow_consistency_mean"] < 0.6]

sme_cde = lf[(lf["product_type"] == "SME") & 
             (lf["origination_risk_grade"].isin(["C", "D", "E"]))]
# L5 cutoff corrected 0.60 -> 0.65 (empirical optimum, WS H §4).
lever5_targets = sme_cde[sme_cde["cashflow_consistency_mean"] < 0.65]

sme_ab = lf[(lf["product_type"] == "SME") & 
            (lf["origination_risk_grade"].isin(["A", "B"]))]
n_new = int(round(len(sme_ab) * 0.10))
mean_vp = sme_ab["value_proxy"].mean()
mean_ticket = sme_ab["ticket_size"].mean()

print(f"Lever 1 targets (Cease BNPL Digital+DSA): {len(lever1_targets):,} loans")
print(f"Lever 2 targets (Tighten Personal D-E):  {len(lever2_targets):,} loans")
print(f"Lever 4 (Grow SME A-B by 10%):           {n_new:,} new loans")
print(f"Lever 5 targets (Contain SME C-D-E):     {len(lever5_targets):,} loans")

# --- Helper functions ---
def compute_cease_net(target_loans, lgd, stress_pp=0):
    """Net Δ in ₹ cr for a cease/tighten lever at given stress level."""
    baseline = -target_loans["value_proxy"].sum() / 1e7
    extra = (target_loans["ticket_size"] * lgd * stress_pp / 100).sum() / 1e7
    return baseline + extra

def compute_grow_net(n_loans, mean_vp_inr, mean_ticket_inr, lgd, stress_pp=0):
    """Net Δ in ₹ cr for a grow lever at given stress level."""
    baseline = n_loans * mean_vp_inr / 1e7
    stress_loss = n_loans * mean_ticket_inr * lgd * stress_pp / 100 / 1e7
    return baseline - stress_loss

Loaded 49,600 loans with segments

Lever 1 targets (Cease BNPL Digital+DSA): 8,610 loans
Lever 2 targets (Tighten Personal D-E):  767 loans
Lever 4 (Grow SME A-B by 10%):           456 new loans
Lever 5 targets (Contain SME C-D-E):     950 loans


In [2]:
# Cell 2 (WS H) — Extended macro stress on combined portfolio
# Tests the recommendation at 0pp through +5pp default rate stress.

stress_levels = [0, 1, 2, 3, 4, 5]
rows = []
for s in stress_levels:
    l1 = compute_cease_net(lever1_targets, LGD_BNPL, s)
    l2 = compute_cease_net(lever2_targets, LGD_PERSONAL, s)
    l4 = compute_grow_net(n_new, mean_vp, mean_ticket, LGD_SME, s)
    l5 = compute_cease_net(lever5_targets, LGD_SME, s)
    combined = l1 + l2 + l4 + l5
    rows.append({
        "Stress (pp)": s,
        "L1 (Cease BNPL)": round(l1, 2),
        "L2 (Tighten Personal)": round(l2, 2),
        "L4 (Grow SME A-B)": round(l4, 2),
        "L5 (Contain SME C-D-E)": round(l5, 2),
        "Combined": round(combined, 2),
    })

stress_extended = pd.DataFrame(rows)
print("Extended macro stress sensitivity — Net Δ in ₹ cr:\n")
print(stress_extended.to_string(index=False))

# Save to WS H folder
stress_extended.to_csv("extended_stress.csv", index=False)
print("\nSaved: extended_stress.csv (in WS H folder)")

Extended macro stress sensitivity — Net Δ in ₹ cr:

 Stress (pp)  L1 (Cease BNPL)  L2 (Tighten Personal)  L4 (Grow SME A-B)  L5 (Contain SME C-D-E)  Combined
           0             1.08                   2.33               1.47                   12.97     17.85
           1             1.13                   2.39               1.08                   13.64     18.24
           2             1.18                   2.45               0.69                   14.30     18.62
           3             1.24                   2.51               0.30                   14.96     19.01
           4             1.29                   2.57              -0.09                   15.62     19.40
           5             1.34                   2.63              -0.48                   16.29     19.78

Saved: extended_stress.csv (in WS H folder)


In [3]:
# Cell 3 (WS H) — LGD sensitivity at the locked +3pp stress level
# LGD only affects the stress component, so we evaluate where it matters most.
# Flex all three product LGDs jointly around the locked, data-generating values
# (BNPL 40% / Personal 65% / SME 75%, config.py A-044).

lgd_scenarios = [
    ("LGD −10pp (optimistic)", -0.10),
    ("LGD −5pp",                -0.05),
    ("Locked (baseline)",        0.00),
    ("LGD +5pp",                 0.05),
    ("LGD +10pp (pessimistic)",  0.10),
]

rows = []
for label, delta in lgd_scenarios:
    bnpl_lgd, ppl_lgd, sme_lgd = LGD_BNPL + delta, LGD_PERSONAL + delta, LGD_SME + delta
    l1 = compute_cease_net(lever1_targets, bnpl_lgd, stress_pp=3)
    l2 = compute_cease_net(lever2_targets, ppl_lgd, stress_pp=3)
    l4 = compute_grow_net(n_new, mean_vp, mean_ticket, sme_lgd, stress_pp=3)
    l5 = compute_cease_net(lever5_targets, sme_lgd, stress_pp=3)
    combined = l1 + l2 + l4 + l5
    rows.append({
        "Scenario": label,
        "BNPL LGD": f"{int(round(bnpl_lgd*100))}%",
        "Personal LGD": f"{int(round(ppl_lgd*100))}%",
        "SME LGD": f"{int(round(sme_lgd*100))}%",
        "L1": round(l1, 2),
        "L2": round(l2, 2),
        "L4": round(l4, 2),
        "L5": round(l5, 2),
        "Combined": round(combined, 2),
    })

lgd_sens = pd.DataFrame(rows)
print("LGD sensitivity at +3pp stress — Net Δ in ₹ cr:\n")
print(lgd_sens.to_string(index=False))

lgd_sens.to_csv("lgd_sensitivity.csv", index=False)
print("\nSaved: lgd_sensitivity.csv (in WS H folder)")

LGD sensitivity at +3pp stress — Net Δ in ₹ cr:

               Scenario BNPL LGD Personal LGD SME LGD   L1   L2   L4    L5  Combined
 LGD −10pp (optimistic)      30%          55%     65% 1.20 2.48 0.46 14.70     18.83
               LGD −5pp      35%          60%     70% 1.22 2.50 0.38 14.83     18.92
      Locked (baseline)      40%          65%     75% 1.24 2.51 0.30 14.96     19.01
               LGD +5pp      45%          70%     80% 1.26 2.53 0.22 15.09     19.10
LGD +10pp (pessimistic)      50%          75%     85% 1.27 2.54 0.15 15.23     19.19

Saved: lgd_sensitivity.csv (in WS H folder)


In [4]:
# Cell 4 (WS H) — Cashflow cutoff sensitivity for L2 (Personal D-E) and L5 (SME C-D-E)
# Tests cutoffs from 0.50 to 0.70; we expect impact to peak at 0.60.

cutoffs = [0.50, 0.55, 0.60, 0.65, 0.70]

rows = []
for c in cutoffs:
    l2_targets_c = ppl_de[ppl_de["cashflow_consistency_mean"] < c]
    l5_targets_c = sme_cde[sme_cde["cashflow_consistency_mean"] < c]
    
    rows.append({
        "Cutoff": c,
        "L2 loans": len(l2_targets_c),
        "L2 baseline": round(compute_cease_net(l2_targets_c, LGD_PERSONAL, 0), 2),
        "L2 at +3pp": round(compute_cease_net(l2_targets_c, LGD_PERSONAL, 3), 2),
        "L5 loans": len(l5_targets_c),
        "L5 baseline": round(compute_cease_net(l5_targets_c, LGD_SME, 0), 2),
        "L5 at +3pp": round(compute_cease_net(l5_targets_c, LGD_SME, 3), 2),
    })

cutoff_sens = pd.DataFrame(rows)
print("Cashflow cutoff sensitivity — Net Δ in ₹ cr:\n")
print(cutoff_sens.to_string(index=False))

cutoff_sens.to_csv("cutoff_sensitivity.csv", index=False)
print("\nSaved: cutoff_sensitivity.csv (in WS H folder)")

Cashflow cutoff sensitivity — Net Δ in ₹ cr:

 Cutoff  L2 loans  L2 baseline  L2 at +3pp  L5 loans  L5 baseline  L5 at +3pp
   0.50       210         0.82        0.87       151         4.04        4.35
   0.55       395         1.50        1.59       265         6.75        7.30
   0.60       767         2.33        2.51       498        10.61       11.63
   0.65      1409         2.42        2.75       950        12.97       14.96
   0.70      2935         0.73        1.43      2035         6.05       10.28

Saved: cutoff_sensitivity.csv (in WS H folder)


In [5]:
# Cell 5 (WS H) — L4 growth rate sensitivity
# Tests how the Grow SME A-B lever scales with growth rate from 5% to 20%.

growth_rates = [5, 10, 15, 20]

rows = []
for g in growth_rates:
    n = int(round(len(sme_ab) * g / 100))
    baseline = compute_grow_net(n, mean_vp, mean_ticket, LGD_SME, 0)
    stress_3pp = compute_grow_net(n, mean_vp, mean_ticket, LGD_SME, 3)
    rows.append({
        "Growth %": g,
        "New loans": n,
        "Origination added (cr)": round(n * mean_ticket / 1e7, 2),
        "Baseline (cr)": round(baseline, 2),
        "+3pp stress (cr)": round(stress_3pp, 2),
        "Δ baseline→+3pp": round(stress_3pp - baseline, 2),
    })

l4_sens = pd.DataFrame(rows)
print("L4 growth rate sensitivity — Net Δ in ₹ cr:\n")
print(l4_sens.to_string(index=False))

l4_sens.to_csv("growth_rate_sensitivity.csv", index=False)
print("\nSaved: growth_rate_sensitivity.csv (in WS H folder)")

L4 growth rate sensitivity — Net Δ in ₹ cr:

 Growth %  New loans  Origination added (cr)  Baseline (cr)  +3pp stress (cr)  Δ baseline→+3pp
        5        228                   25.92           0.73              0.15            -0.58
       10        456                   51.83           1.47              0.30            -1.17
       15        684                   77.75           2.20              0.45            -1.75
       20        912                  103.67           2.94              0.60            -2.33

Saved: growth_rate_sensitivity.csv (in WS H folder)
